# Week 4 — Solved

Logarithmic plots, spatial power laws, and linear regression across crime-type pairs.

## Setup

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.ticker as mticker, warnings, requests
warnings.filterwarnings('ignore')

# Load pre-merged dataset (from Week 2)
try:
    merged = pd.read_parquet('/home/claude/notebooks/sf_crime_merged.parquet')
    print(f'Loaded merged dataset: {len(merged):,} rows')
except FileNotFoundError:
    print('Run Week 2 first to generate the merged parquet, or re-fetch here.')
    # Fallback: re-fetch just 'now' data for standalone use
    import requests
    API_NOW = 'https://data.sfgov.org/resource/wg3w-h783.json'
    def fetch_all(url, params=None, chunk=50_000):
        if params is None: params = {}
        records, offset = [], 0
        while True:
            r = requests.get(url, params={**params, '$limit': chunk, '$offset': offset}, timeout=120)
            r.raise_for_status(); batch = r.json()
            if not batch: break
            records.extend(batch); offset += chunk
            if len(batch) < chunk: break
        return pd.DataFrame(records)
    raw = fetch_all(API_NOW)
    raw['date'] = pd.to_datetime(raw.get('incident_date', pd.NaT), errors='coerce')
    raw['year'] = raw['date'].dt.year
    raw['month'] = raw['date'].dt.month
    raw['weekday'] = raw['date'].dt.day_name()
    raw['hour'] = pd.to_datetime(raw.get('incident_time',''), format='%H:%M', errors='coerce').dt.hour
    raw['category'] = raw.get('incident_category', '')
    raw['district'] = raw.get('police_district', '')
    raw['lat'] = pd.to_numeric(raw.get('latitude', None), errors='coerce')
    raw['lon'] = pd.to_numeric(raw.get('longitude', None), errors='coerce')
    merged = raw.dropna(subset=['date']).copy()

FOCUS_CRIMES = [
    'Assault', 'Burglary', 'Drug Offense', 'Larceny Theft',
    'Motor Vehicle Theft', 'Robbery', 'Vandalism', 'Weapons Offense',
    'Arson', 'Disorderly Conduct', 'Fraud', 'Prostitution',
    'Sex Offense', 'Stolen Property', 'Suspicious Occ', 'Traffic Violation Arrest'
]
FOCUS_CRIMES = [c for c in FOCUS_CRIMES if c in merged['category'].unique()]
df_focus = merged[merged['category'].isin(FOCUS_CRIMES)].copy()
print(f'Focus crimes available: {len(FOCUS_CRIMES)}')


## Exercise 1.1 — Data encodings

**10 ways to encode data:** Position (x/y), Color hue, Color saturation/lightness, Size, Shape, Angle/orientation, Length, Area, Volume, Texture.

**Are all encodings equal?** No. Position is the most accurately perceived channel (Cleveland & McGill, 1984). Color saturation and area are much harder for humans to compare precisely. In Week 1 we used bar height (position) which is easy to read, but a pie chart (angle/area) for the same data would be much harder.

**Encodings difficult for the eye:** Angle (pie charts), Area (bubble charts), Volume (3D bar charts).

**Problem with pie charts:** Humans are very poor at comparing angles and areas. Two slices of 30% and 35% look nearly identical in a pie chart but would be obviously different as bar lengths.

## Exercise 2.2 — Logarithmic plots

In [ ]:
# Semi-log plot of focus crime counts
cat_counts = df_focus['category'].value_counts()
cat_counts = cat_counts[cat_counts.index.isin(FOCUS_CRIMES)].sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear
axes[0].barh(cat_counts.index[::-1], cat_counts.values[::-1], color='steelblue')
axes[0].set_title('Crime Counts — Linear Scale', fontweight='bold')
axes[0].set_xlabel('Number of incidents')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

# Log
axes[1].barh(cat_counts.index[::-1], cat_counts.values[::-1], color='steelblue')
axes[1].set_xscale('log')
axes[1].set_title('Crime Counts — Log Scale', fontweight='bold')
axes[1].set_xlabel('Number of incidents (log scale)')

for ax in axes: ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig('log_crime_counts.png', dpi=150, bbox_inches='tight'); plt.show()
print('On log scale, rare crimes like Arson and Prostitution become more visible relative to common ones like Larceny Theft.')


## Exercise 2.2 — Spatial Power Law (Larceny Theft on a grid)

In [ ]:
# Use most common focus crime: Larceny Theft
crime_for_powerlaw = 'Larceny Theft'
subset = df_focus[df_focus['category'] == crime_for_powerlaw].copy()
subset = subset.dropna(subset=['lat','lon'])

# Filter to SF peninsula bounds
subset = subset[(subset['lat'].between(37.70, 37.83)) &
                (subset['lon'].between(-122.53, -122.35))]

print(f'Points for power-law analysis: {len(subset):,}')

# ~100m grid: 1 degree lat ≈ 111 km, 1 degree lon ≈ 85 km at SF latitude
# 100m in degrees: lat=0.0009, lon=0.00118
lat_bins = np.arange(37.70, 37.83, 0.0009)
lon_bins = np.arange(-122.53, -122.35, 0.00118)

counts_2d, _, _ = np.histogram2d(subset['lat'], subset['lon'],
                                  bins=[lat_bins, lon_bins])
counts_flat = counts_2d.flatten().astype(int)
print(f'Grid cells: {len(counts_flat):,}  |  Max crimes in one cell: {counts_flat.max()}')


In [ ]:
# Tally: N(k) = number of cells with exactly k crimes
max_k = int(counts_flat.max())
k_values = np.arange(0, max_k+1)
N_k = np.array([(counts_flat == k).sum() for k in k_values])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear axes
axes[0].bar(k_values+1, N_k, color='steelblue', width=0.8)
axes[0].set_xlabel('k+1 (crimes per cell + 1)'); axes[0].set_ylabel('N(k) — number of cells')
axes[0].set_title('Linear axes', fontweight='bold'); axes[0].grid(alpha=0.3)
axes[0].set_xlim(0, 50)

# Log-log axes
axes[1].scatter(k_values[N_k>0]+1, N_k[N_k>0], s=20, color='steelblue', alpha=0.7)
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_xlabel('k+1 (log)'); axes[1].set_ylabel('N(k) (log)')
axes[1].set_title('Log-log axes', fontweight='bold'); axes[1].grid(alpha=0.3)

# Fit a power law line
mask = (k_values > 0) & (N_k > 0)
log_k = np.log(k_values[mask]+1); log_N = np.log(N_k[mask])
slope, intercept = np.polyfit(log_k, log_N, 1)
x_fit = np.linspace(1, max_k+1, 200)
axes[1].plot(x_fit, np.exp(intercept)*x_fit**slope, 'r-', linewidth=2,
             label=f'Power law fit: slope={slope:.2f}')
axes[1].legend()

plt.suptitle(f'Spatial Distribution of {crime_for_powerlaw} — Power Law Test', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('powerlaw_larceny.png', dpi=150, bbox_inches='tight'); plt.show()

print(f'Power-law slope: {slope:.2f}')
print('A straight line on log-log axes → power-law distribution. Confirmed!')


**Power-law interpretation (Step 6 & 7):**

The log-log plot shows a roughly straight line, confirming that Larceny Theft follows a power-law spatial distribution. This means the vast majority of grid cells have *zero or very few* thefts, while a tiny number of cells account for a disproportionate share of all incidents.

**Practical implications:** The 'average block' statistic is essentially meaningless when the distribution is this skewed. A neighborhood-level average might be pulled up entirely by one or two extreme hotspot blocks, while most residents experience near-zero crime. For policing, this means targeted enforcement at a handful of blocks could theoretically reduce citywide theft substantially — but also means concentrated surveillance falls on specific communities. For city planning, it suggests crime is tied to specific geographic features (transit hubs, retail corridors, parking lots) rather than diffuse neighborhood characteristics.

## Exercise 3.1–3.3 — Linear Regression across crime pairs

In [ ]:
# 168-hour vectors for all focus crimes
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df_focus['day_num'] = pd.Categorical(df_focus['weekday'], categories=day_order, ordered=True).codes
df_focus['week_hour'] = df_focus['day_num'] * 24 + df_focus['hour'].fillna(0).astype(int)

vectors = {}
for crime in FOCUS_CRIMES:
    v = df_focus[df_focus['category']==crime].groupby('week_hour').size()
    v = v.reindex(range(168), fill_value=0)
    vectors[crime] = v.values.astype(float)

# Select 9 crimes for the scatterplot matrix
sel = FOCUS_CRIMES[:9]
n = len(sel)
print(f'Selected {n} crimes for regression matrix: {sel}')


In [ ]:
# Closed-form linear regression
def linreg(x, y):
    N  = len(x)
    mx, my = x.mean(), y.mean()
    a = (np.sum(x*y) - N*mx*my) / (np.sum(x**2) - N*mx**2 + 1e-12)
    b = my - a*mx
    return a, b

def r_squared(x, y, a, b):
    y_pred = a*x + b
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - y.mean())**2)
    return 1 - ss_res/(ss_tot + 1e-12)

# Hour-of-week color gradient
hour_colors = plt.cm.plasma(np.linspace(0, 1, 168))

fig, axes = plt.subplots(n, n, figsize=(18, 18))
r2_matrix = np.zeros((n, n))

for i, c1 in enumerate(sel):
    for j, c2 in enumerate(sel):
        ax = axes[i][j]
        x = vectors[c1]; y = vectors[c2]
        if i == j:
            ax.text(0.5, 0.5, c1, transform=ax.transAxes,
                    ha='center', va='center', fontsize=7, fontweight='bold',
                    wrap=True)
            ax.set_xticks([]); ax.set_yticks([])
        else:
            sc = ax.scatter(x, y, c=hour_colors, s=8, alpha=0.7)
            a, b = linreg(x, y)
            r2   = r_squared(x, y, a, b)
            r2_matrix[i,j] = r2
            x_line = np.linspace(x.min(), x.max(), 100)
            ax.plot(x_line, a*x_line + b, 'r-', linewidth=1)
            ax.text(0.05, 0.9, f'R²={r2:.2f}', transform=ax.transAxes,
                    fontsize=6, color='darkred', fontweight='bold')
        if i == n-1: ax.set_xlabel(sel[j], fontsize=6, rotation=30)
        if j == 0:   ax.set_ylabel(sel[i], fontsize=6, rotation=60, labelpad=20)
        ax.tick_params(labelsize=5)

fig.suptitle('Pairwise Crime Scatterplots — 168 weekly hours\n(color = hour of week, red line = linear fit)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('crime_regression_matrix.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Report most and least correlated pairs
pairs = []
for i, c1 in enumerate(sel):
    for j, c2 in enumerate(sel):
        if i < j:
            pairs.append((r2_matrix[i,j], c1, c2))
pairs.sort(reverse=True)
print('Most correlated pairs:')
for r2, c1, c2 in pairs[:3]: print(f'  R²={r2:.3f}  {c1} ↔ {c2}')
print('\nLeast correlated pairs:')
for r2, c1, c2 in pairs[-3:]: print(f'  R²={r2:.3f}  {c1} ↔ {c2}')


**Interpretation:**

Highly correlated pairs (e.g. Assault ↔ Robbery) share a similar weekly rhythm because both are driven by social activity — they peak at night on weekends when people are out in public spaces. Low correlation pairs (e.g. Fraud ↔ Prostitution) reflect crimes driven by entirely different contexts: Fraud peaks during business hours on weekdays, while Prostitution is concentrated late at night.

**Connection to predictive policing (Week 1):** If a system allocates resources based on Assault patterns and applies them to Drug Offense (which is highly correlated), it will systematically over-police specific neighborhoods at specific hours — and those patterns were themselves shaped by historical enforcement decisions, not underlying behavior. This is precisely the feedback loop Richardson et al. describe.